In [0]:
%pip install openpyxl xlrd

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
import os
import re
from datetime import datetime, timezone

SOURCE_ROOT = "/Volumes/cmu_hackathon/common/unocha"
TARGET_SCHEMA = "workspace.default"
TARGET_PREFIX = "njyoti_bronze_unocha"
MANIFEST_TABLE = f"{TARGET_SCHEMA}.{TARGET_PREFIX}_manifest"
SHEET_MANIFEST_TABLE = f"{TARGET_SCHEMA}.{TARGET_PREFIX}_sheet_manifest"
MAX_TABLE_NAME_LENGTH = 255

def to_local_volume_path(path: str) -> str:
    return path.replace("dbfs:/Volumes/", "/Volumes/") if path.startswith("dbfs:/Volumes/") else path

In [0]:
file_rows = []
for file_info in dbutils.fs.ls(SOURCE_ROOT):
    if file_info.isDir():
        continue
    file_name = os.path.basename(file_info.path.rstrip("/"))
    extension = file_name.rsplit(".", 1)[-1].lower() if "." in file_name else ""
    file_rows.append({
        "source_path": file_info.path,
        "file_name": file_name,
        "extension": extension,
        "size_bytes": int(file_info.size),
        "discovered_at": datetime.now(timezone.utc).isoformat()
    })

files_df = spark.createDataFrame(file_rows)
display(files_df.orderBy("extension", "file_name"))

In [0]:
(
    files_df
    .withColumn("ingested_at", F.current_timestamp())
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(MANIFEST_TABLE)
)

display(spark.table(MANIFEST_TABLE).orderBy("extension", "file_name"))

In [0]:
import pandas as pd

excel_extensions = {"xls", "xlsx"}
sheet_rows = []

for row in files_df.collect():
    if row["extension"] not in excel_extensions:
        continue
    workbook_path = to_local_volume_path(row["source_path"])
    workbook = pd.ExcelFile(workbook_path)
    for position, sheet_name in enumerate(workbook.sheet_names, start=1):
        sheet_rows.append({
            "source_path": row["source_path"],
            "file_name": row["file_name"],
            "extension": row["extension"],
            "sheet_name": sheet_name,
            "sheet_position": position,
            "discovered_at": datetime.now(timezone.utc).isoformat()
        })

if sheet_rows:
    sheet_manifest_df = spark.createDataFrame(sheet_rows)
else:
    sheet_manifest_schema = T.StructType([
        T.StructField("source_path", T.StringType(), True),
        T.StructField("file_name", T.StringType(), True),
        T.StructField("extension", T.StringType(), True),
        T.StructField("sheet_name", T.StringType(), True),
        T.StructField("sheet_position", T.IntegerType(), True),
        T.StructField("discovered_at", T.StringType(), True)
    ])
    sheet_manifest_df = spark.createDataFrame([], sheet_manifest_schema)

display(sheet_manifest_df.orderBy("file_name", "sheet_position"))

In [0]:
(
    sheet_manifest_df
    .withColumn("ingested_at", F.current_timestamp())
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SHEET_MANIFEST_TABLE)
)

display(spark.table(SHEET_MANIFEST_TABLE).orderBy("file_name", "sheet_position"))

In [0]:
import pandas as pd

excel_extensions = {"xls", "xlsx"}
non_data_sheet_patterns = [
    r"^introduction$",
    r"^what is",
    r"^table of contents$",
    r"^key figures$",
    r"^log$",
    r"^lists?$",
    r"^indicator metadata$",
    r"^indicator date hidden",
    r"^imputed and missing data hidden$"
]

def normalize_name(value: str) -> str:
    value = str(value).strip().lower()
    value = re.sub(r"[^a-z0-9]+", "_", value)
    value = re.sub(r"_+", "_", value).strip("_")
    return value or "unnamed"

def classify_csv_dataset(file_name: str) -> str:
    base_name = file_name.rsplit(".", 1)[0]
    base_name = re.sub(r"__\d{8}_\d{6}_utc$", "", base_name, flags=re.IGNORECASE)
    return normalize_name(base_name)

def is_data_sheet(sheet_name: str) -> bool:
    normalized_sheet = str(sheet_name).strip().lower()
    for pattern in non_data_sheet_patterns:
        if re.search(pattern, normalized_sheet):
            return False
    return True

def classify_excel_dataset(file_name: str, sheet_name: str) -> str:
    workbook_name = file_name.rsplit(".", 1)[0]
    workbook_name = re.sub(r"^\d+[-_]?", "", workbook_name)
    workbook_name = re.sub(r"(beta_version|dataset|update|final|v\d+)", "", workbook_name, flags=re.IGNORECASE)
    workbook_name = normalize_name(workbook_name)
    sheet_component = normalize_name(sheet_name)
    if sheet_component in {"gcsi", "impact_of_crisis", "conditions_of_affected_people", "complexity", "core_indicators", "crisis_indicator_data", "country_indicator_data"}:
        return f"{workbook_name}_{sheet_component}"
    if "inform_severity" in workbook_name or "acaps_inform_severity" in workbook_name:
        return f"inform_severity_{sheet_component}"
    return f"{workbook_name}_{sheet_component}"

def build_table_name(dataset_name: str) -> str:
    base_name = f"{TARGET_PREFIX}_{normalize_name(dataset_name)}"
    return f"{TARGET_SCHEMA}.{base_name[:MAX_TABLE_NAME_LENGTH]}"

def deduplicate_columns(column_names):
    normalized_cols = []
    seen = set()
    for column_name in column_names:
        normalized = normalize_name(column_name)
        candidate = normalized
        suffix = 1
        while candidate in seen:
            suffix += 1
            candidate = f"{normalized}_{suffix}"
        normalized_cols.append(candidate)
        seen.add(candidate)
    return normalized_cols

def attach_metadata(df, dataset_name: str, source_path: str, file_name: str, sheet_name: str | None = None):
    result = df.toDF(*deduplicate_columns(df.columns))
    return (
        result
        .withColumn("_source_file", F.lit(source_path))
        .withColumn("_source_file_name", F.lit(file_name))
        .withColumn("_sheet_name", F.lit(sheet_name))
        .withColumn("_dataset_name", F.lit(normalize_name(dataset_name)))
        .withColumn("_ingest_ts", F.current_timestamp())
    )

def read_csv_file(source_path: str):
    return (
        spark.read
        .option("header", True)
        .option("multiLine", False)
        .option("escape", '"')
        .csv(source_path)
    )

def read_excel_sheet(source_path: str, sheet_name: str):
    local_path = to_local_volume_path(source_path)
    pdf = pd.read_excel(local_path, sheet_name=sheet_name)
    if pdf.empty:
        return None
    pdf = pdf.dropna(axis=0, how="all").dropna(axis=1, how="all")
    if pdf.empty:
        return None
    pdf.columns = [str(col) for col in pdf.columns]
    pdf = pdf.where(pd.notnull(pdf), None)
    records = pdf.to_dict(orient="records")
    if not records:
        return None
    return spark.createDataFrame(records)

def write_bronze_table(df, table_name: str):
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table_name)
    )

In [0]:
csv_dataset_map = {}
for row in files_df.filter(F.col("extension") == "csv").collect():
    dataset_name = classify_csv_dataset(row["file_name"])
    csv_dataset_map.setdefault(dataset_name, []).append(row.asDict())

csv_table_rows = []
for dataset_name, dataset_files in sorted(csv_dataset_map.items()):
    bronze_df = None
    source_paths = []
    for file_row in dataset_files:
        current_df = read_csv_file(file_row["source_path"])
        current_df = attach_metadata(current_df, dataset_name, file_row["source_path"], file_row["file_name"])
        source_paths.append(file_row["source_path"])
        bronze_df = current_df if bronze_df is None else bronze_df.unionByName(current_df, allowMissingColumns=True)
    table_name = build_table_name(dataset_name)
    write_bronze_table(bronze_df, table_name)
    csv_table_rows.append({
        "dataset_name": dataset_name,
        "table_name": table_name,
        "source_count": len(dataset_files),
        "source_paths": source_paths,
        "record_count": bronze_df.count(),
        "ingested_at": datetime.now(timezone.utc).isoformat(),
        "source_type": "csv"
    })

csv_table_schema = T.StructType([
    T.StructField("dataset_name", T.StringType(), True),
    T.StructField("table_name", T.StringType(), True),
    T.StructField("source_count", T.IntegerType(), True),
    T.StructField("source_paths", T.ArrayType(T.StringType()), True),
    T.StructField("record_count", T.LongType(), True),
    T.StructField("ingested_at", T.StringType(), True),
    T.StructField("source_type", T.StringType(), True)
])

csv_tables_df = spark.createDataFrame(csv_table_rows, schema=csv_table_schema) if csv_table_rows else spark.createDataFrame([], schema=csv_table_schema)
display(csv_tables_df.orderBy("dataset_name"))

In [0]:

#DO NOT RUN OR RE-RUN - THIS WILL DELETE ALL TABLES LOADED FROM EXCEL AND STARTED SERIAL INGESTION TAKING 4+ HOURS
import time

# --- Cleanup any previously created Excel stage/bronze tables ---
existing_tables = spark.sql(f"SHOW TABLES IN {TARGET_SCHEMA}").collect()
for tbl in existing_tables:
    tbl_name = tbl["tableName"]
    if tbl_name.startswith(f"{TARGET_PREFIX}_stage_") or (
        tbl_name.startswith(TARGET_PREFIX) and "inform_severity" in tbl_name and tbl_name not in [
            MANIFEST_TABLE.split(".")[-1],
            SHEET_MANIFEST_TABLE.split(".")[-1]
        ]
    ):
        spark.sql(f"DROP TABLE IF EXISTS {TARGET_SCHEMA}.{tbl_name}")
        print(f"Dropped leftover: {tbl_name}")

# --- Select only the latest workbook per logical dataset ---
SELECTED_EXCEL_SHEETS_TABLE = f"{TARGET_SCHEMA}.{TARGET_PREFIX}_excel_selected_sheets"
EXCEL_STAGE_TABLE_PREFIX = f"{TARGET_PREFIX}_stage"

selected_by_dataset = {}
for row in [r.asDict() for r in sheet_manifest_df.collect() if is_data_sheet(r["sheet_name"])]:
    dataset_name = classify_excel_dataset(row["file_name"], row["sheet_name"])
    candidate = {**row, "dataset_name": dataset_name}
    existing = selected_by_dataset.get(dataset_name)
    if existing is None or candidate["file_name"] > existing["file_name"]:
        selected_by_dataset[dataset_name] = candidate

selected_sheet_rows = sorted(selected_by_dataset.values(), key=lambda x: (x["dataset_name"], x["file_name"], x["sheet_position"]))
print(f"Selected {len(selected_sheet_rows)} sheets across {len(set(r['dataset_name'] for r in selected_sheet_rows))} datasets")

selected_sheet_schema = T.StructType([
    T.StructField("dataset_name", T.StringType(), True),
    T.StructField("source_path", T.StringType(), True),
    T.StructField("file_name", T.StringType(), True),
    T.StructField("extension", T.StringType(), True),
    T.StructField("sheet_name", T.StringType(), True),
    T.StructField("sheet_position", T.LongType(), True),
    T.StructField("discovered_at", T.StringType(), True)
])
selected_sheets_df = spark.createDataFrame(selected_sheet_rows, schema=selected_sheet_schema) if selected_sheet_rows else spark.createDataFrame([], schema=selected_sheet_schema)

(
    selected_sheets_df
    .withColumn("selected_at", F.current_timestamp())
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SELECTED_EXCEL_SHEETS_TABLE)
)

# --- Stage each selected sheet with retry logic ---
def build_stage_table_name(dataset_name: str) -> str:
    base_name = f"{EXCEL_STAGE_TABLE_PREFIX}_{normalize_name(dataset_name)}"
    return f"{TARGET_SCHEMA}.{base_name[:MAX_TABLE_NAME_LENGTH]}"

def stage_selected_sheet(sheet_row: dict, max_retries: int = 3):
    local_path = to_local_volume_path(sheet_row["source_path"])
    pdf = pd.read_excel(local_path, sheet_name=sheet_row["sheet_name"], dtype=str)
    if pdf.empty:
        return None, 0
    pdf = pdf.dropna(axis=0, how="all").dropna(axis=1, how="all")
    if pdf.empty:
        return None, 0
    pdf.columns = deduplicate_columns([str(col) for col in pdf.columns])
    pdf = pdf.where(pd.notnull(pdf), None)

    table_name = build_stage_table_name(sheet_row["dataset_name"])
    schema = T.StructType([T.StructField(col, T.StringType(), True) for col in pdf.columns])
    rows = [tuple(None if pd.isna(v) else str(v) for v in row) for row in pdf.itertuples(index=False, name=None)]
    if not rows:
        return None, 0

    for attempt in range(max_retries):
        try:
            sdf = spark.createDataFrame(rows, schema=schema)
            sdf = attach_metadata(
                sdf,
                sheet_row["dataset_name"],
                sheet_row["source_path"],
                sheet_row["file_name"],
                sheet_row["sheet_name"]
            )
            sdf.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(table_name)
            return table_name, len(rows)
        except Exception as e:
            if "TEMPORARILY_UNAVAILABLE" in str(e) and attempt < max_retries - 1:
                wait_time = 5 * (attempt + 1)
                print(f"  -> Service unavailable, retrying in {wait_time}s (attempt {attempt + 1}/{max_retries})...")
                time.sleep(wait_time)
            else:
                raise
    return None, 0

excel_table_rows = []
failed_sheets = []
for i, sheet_row in enumerate(selected_sheet_rows, 1):
    if i % 10 == 0:
        print(f"[{i}/{len(selected_sheet_rows)}] Processing...")
    try:
        table_name, record_count = stage_selected_sheet(sheet_row)
        if table_name is None:
            continue
        excel_table_rows.append({
            "dataset_name": sheet_row["dataset_name"],
            "table_name": table_name,
            "source_count": 1,
            "source_paths": [f"{sheet_row['file_name']}::{sheet_row['sheet_name']}"],
            "record_count": record_count,
            "ingested_at": datetime.now(timezone.utc).isoformat(),
            "source_type": "excel_stage"
        })
    except Exception as e:
        failed_sheets.append({"dataset": sheet_row["dataset_name"], "error": str(e)[:200]})
        print(f"  -> FAILED: {sheet_row['dataset_name']}: {str(e)[:100]}")
    time.sleep(0.5)  # throttle to avoid overwhelming Spark Connect

print(f"\nDone: {len(excel_table_rows)} staged, {len(failed_sheets)} failed")

excel_table_schema = T.StructType([
    T.StructField("dataset_name", T.StringType(), True),
    T.StructField("table_name", T.StringType(), True),
    T.StructField("source_count", T.IntegerType(), True),
    T.StructField("source_paths", T.ArrayType(T.StringType()), True),
    T.StructField("record_count", T.LongType(), True),
    T.StructField("ingested_at", T.StringType(), True),
    T.StructField("source_type", T.StringType(), True)
])
excel_tables_df = spark.createDataFrame(excel_table_rows, schema=excel_table_schema) if excel_table_rows else spark.createDataFrame([], schema=excel_table_schema)
display(excel_tables_df.orderBy("dataset_name"))

In [0]:
#DO NOT RUN OR RERUN - HIGH RISK OF DATA CORRUPTION IN BRONZE
import time

# --- Reconstruct constants and helpers from cell 9 ---
EXCEL_STAGE_TABLE_PREFIX = f"{TARGET_PREFIX}_stage"

def build_stage_table_name(dataset_name: str) -> str:
    base_name = f"{EXCEL_STAGE_TABLE_PREFIX}_{normalize_name(dataset_name)}"
    return f"{TARGET_SCHEMA}.{base_name[:MAX_TABLE_NAME_LENGTH]}"

# --- Reconstruct selected_sheet_rows from sheet_manifest_df (loaded in cell 5) ---
selected_by_dataset = {}
for row in [r.asDict() for r in sheet_manifest_df.collect() if is_data_sheet(r["sheet_name"])]:
    dataset_name = classify_excel_dataset(row["file_name"], row["sheet_name"])
    candidate = {**row, "dataset_name": dataset_name}
    existing = selected_by_dataset.get(dataset_name)
    if existing is None or candidate["file_name"] > existing["file_name"]:
        selected_by_dataset[dataset_name] = candidate

selected_sheet_rows = sorted(selected_by_dataset.values(), key=lambda x: (x["dataset_name"], x["file_name"], x["sheet_position"]))
print(f"Total selected sheets: {len(selected_sheet_rows)}")

# --- Check which stage tables already exist ---
existing_tables = {row["tableName"] for row in spark.sql(f"SHOW TABLES IN {TARGET_SCHEMA}").collect()}

missing_sheet_rows = []
already_done = []
for sheet_row in selected_sheet_rows:
    expected_table = build_stage_table_name(sheet_row["dataset_name"]).split(".")[-1]
    if expected_table in existing_tables:
        already_done.append(sheet_row)
    else:
        missing_sheet_rows.append(sheet_row)

print(f"Already ingested: {len(already_done)}")
print(f"Still missing (to retry): {len(missing_sheet_rows)}")

# --- Retry only missing datasets with 10s throttle and 6 retries ---
retry_succeeded = []
retry_failed = []
for i, sheet_row in enumerate(missing_sheet_rows, 1):
    if i % 5 == 0 or i == 1:
        print(f"  [{i}/{len(missing_sheet_rows)}] Processing {sheet_row['dataset_name']}...")
    try:
        local_path = to_local_volume_path(sheet_row["source_path"])
        pdf = pd.read_excel(local_path, sheet_name=sheet_row["sheet_name"], dtype=str)
        if pdf.empty:
            continue
        pdf = pdf.dropna(axis=0, how="all").dropna(axis=1, how="all")
        if pdf.empty:
            continue
        pdf.columns = deduplicate_columns([str(col) for col in pdf.columns])
        pdf = pdf.where(pd.notnull(pdf), None)

        table_name = build_stage_table_name(sheet_row["dataset_name"])
        schema = T.StructType([T.StructField(col, T.StringType(), True) for col in pdf.columns])
        rows = [tuple(None if pd.isna(v) else str(v) for v in row) for row in pdf.itertuples(index=False, name=None)]
        if not rows:
            continue

        max_retries = 6
        success = False
        for attempt in range(max_retries):
            try:
                sdf = spark.createDataFrame(rows, schema=schema)
                sdf = attach_metadata(
                    sdf,
                    sheet_row["dataset_name"],
                    sheet_row["source_path"],
                    sheet_row["file_name"],
                    sheet_row["sheet_name"]
                )
                sdf.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(table_name)
                success = True
                break
            except Exception as e:
                if "TEMPORARILY_UNAVAILABLE" in str(e) and attempt < max_retries - 1:
                    wait_time = 10 * (attempt + 1)
                    print(f"    -> Retry {attempt + 1}/{max_retries} for {sheet_row['dataset_name']}, waiting {wait_time}s...")
                    time.sleep(wait_time)
                else:
                    raise

        if success:
            retry_succeeded.append({
                "dataset_name": sheet_row["dataset_name"],
                "table_name": table_name,
                "source_count": 1,
                "source_paths": [f"{sheet_row['file_name']}::{sheet_row['sheet_name']}"],
                "record_count": len(rows),
                "ingested_at": datetime.now(timezone.utc).isoformat(),
                "source_type": "excel_stage"
            })
    except Exception as e:
        retry_failed.append({"dataset": sheet_row["dataset_name"], "error": str(e)[:200]})
        print(f"    -> FAILED: {sheet_row['dataset_name']}: {str(e)[:100]}")
    time.sleep(10)  # 10s throttle for fresh session

print(f"\nRetry complete: {len(retry_succeeded)} recovered, {len(retry_failed)} still failing")

# --- Build complete excel_tables_df from ALL staged tables (existing + new) ---
excel_table_schema = T.StructType([
    T.StructField("dataset_name", T.StringType(), True),
    T.StructField("table_name", T.StringType(), True),
    T.StructField("source_count", T.IntegerType(), True),
    T.StructField("source_paths", T.ArrayType(T.StringType()), True),
    T.StructField("record_count", T.LongType(), True),
    T.StructField("ingested_at", T.StringType(), True),
    T.StructField("source_type", T.StringType(), True)
])

all_excel_table_rows = []
for sheet_row in selected_sheet_rows:
    tbl_name = build_stage_table_name(sheet_row["dataset_name"])
    short_name = tbl_name.split(".")[-1]
    if short_name in existing_tables or tbl_name in [r["table_name"] for r in retry_succeeded]:
        all_excel_table_rows.append({
            "dataset_name": sheet_row["dataset_name"],
            "table_name": tbl_name,
            "source_count": 1,
            "source_paths": [f"{sheet_row['file_name']}::{sheet_row['sheet_name']}"],
            "record_count": -1,  # placeholder, table already exists
            "ingested_at": datetime.now(timezone.utc).isoformat(),
            "source_type": "excel_stage"
        })

# Override with actual counts from retry_succeeded
for r in retry_succeeded:
    for row in all_excel_table_rows:
        if row["table_name"] == r["table_name"]:
            row["record_count"] = r["record_count"]
            break

excel_tables_df = spark.createDataFrame(all_excel_table_rows, schema=excel_table_schema) if all_excel_table_rows else spark.createDataFrame([], schema=excel_table_schema)
print(f"\nTotal Excel tables: {len(all_excel_table_rows)}")

if retry_failed:
    print("\nPersistently failed datasets:")
    for f in retry_failed:
        print(f"  - {f['dataset']}: {f['error'][:100]}")

display(excel_tables_df.orderBy("dataset_name"))

In [0]:
# Reconstruct csv_tables_df from existing catalog tables (no re-ingestion)
csv_table_schema = T.StructType([
    T.StructField("dataset_name", T.StringType(), True),
    T.StructField("table_name", T.StringType(), True),
    T.StructField("source_count", T.IntegerType(), True),
    T.StructField("source_paths", T.ArrayType(T.StringType()), True),
    T.StructField("record_count", T.LongType(), True),
    T.StructField("ingested_at", T.StringType(), True),
    T.StructField("source_type", T.StringType(), True)
])

skip_suffixes = {"_manifest", "_sheet_manifest", "_excel_selected_sheets"}
all_tables = {row["tableName"] for row in spark.sql(f"SHOW TABLES IN {TARGET_SCHEMA}").collect()}

csv_table_rows = []
for tbl in sorted(all_tables):
    if not tbl.startswith(TARGET_PREFIX):
        continue
    if "_stage_" in tbl:
        continue
    if any(tbl.endswith(s) for s in skip_suffixes):
        continue
    full_name = f"{TARGET_SCHEMA}.{tbl}"
    dataset_name = tbl.replace(f"{TARGET_PREFIX}_", "", 1)
    row_count = spark.table(full_name).count()
    csv_table_rows.append({
        "dataset_name": dataset_name,
        "table_name": full_name,
        "source_count": 1,
        "source_paths": [],
        "record_count": row_count,
        "ingested_at": None,
        "source_type": "csv"
    })

csv_tables_df = spark.createDataFrame(csv_table_rows, schema=csv_table_schema) if csv_table_rows else spark.createDataFrame([], schema=csv_table_schema)
print(f"Reconstructed csv_tables_df from catalog: {len(csv_table_rows)} tables")

# Union with excel
bronze_outputs_df = csv_tables_df.unionByName(excel_tables_df, allowMissingColumns=True)
display(bronze_outputs_df.orderBy("dataset_name"))

In [0]:
%sql
SELECT file_name, sheet_name, sheet_position
FROM workspace.default.njyoti_bronze_unocha_sheet_manifest
WHERE lower(file_name) LIKE '%inform_severity%'
   OR lower(file_name) LIKE '%inform severity%'
ORDER BY file_name, sheet_position

##Troubleshooting Deduping issue for severity data

In [0]:
%sql
SELECT dataset_name, file_name, sheet_name
FROM workspace.default.njyoti_bronze_unocha_excel_selected_sheets
WHERE dataset_name LIKE '%inform_severity_country%'
ORDER BY dataset_name

In [0]:
%sql
-- All INFORM sheets available per workbook vs. what was selected
SELECT 
    m.file_name,
    m.sheet_name,
    m.sheet_position,
    CASE WHEN s.dataset_name IS NOT NULL THEN 'YES' ELSE 'NO' END AS was_selected,
    s.dataset_name
FROM workspace.default.njyoti_bronze_unocha_sheet_manifest m
LEFT JOIN workspace.default.njyoti_bronze_unocha_excel_selected_sheets s
    ON m.file_name = s.file_name AND m.sheet_name = s.sheet_name
WHERE lower(m.file_name) LIKE '%inform_severity%'
ORDER BY m.file_name, m.sheet_position

In [0]:
# Find ALL datasets where multiple workbooks collapsed into one
from collections import defaultdict
import re

# --- Idempotent setup: define constants and helpers if not already available ---
SOURCE_ROOT = "/Volumes/cmu_hackathon/common/unocha"
TARGET_SCHEMA = "workspace.default"
TARGET_PREFIX = "njyoti_bronze_unocha"
SHEET_MANIFEST_TABLE = f"{TARGET_SCHEMA}.{TARGET_PREFIX}_sheet_manifest"

def normalize_name(value: str) -> str:
    value = str(value).strip().lower()
    value = re.sub(r"[^a-z0-9]+", "_", value)
    value = re.sub(r"_+", "_", value).strip("_")
    return value or "unnamed"

non_data_sheet_patterns = [
    r"^introduction$",
    r"^what is",
    r"^table of contents$",
    r"^key figures$",
    r"^log$",
    r"^lists?$",
    r"^indicator metadata$",
    r"^indicator date hidden",
    r"^imputed and missing data hidden$"
]

def is_data_sheet(sheet_name: str) -> bool:
    normalized_sheet = str(sheet_name).strip().lower()
    for pattern in non_data_sheet_patterns:
        if re.search(pattern, normalized_sheet):
            return False
    return True

def classify_excel_dataset(file_name: str, sheet_name: str) -> str:
    workbook_name = file_name.rsplit(".", 1)[0]
    workbook_name = re.sub(r"^\d+[-_]?", "", workbook_name)
    workbook_name = re.sub(r"(beta_version|dataset|update|final|v\d+)", "", workbook_name, flags=re.IGNORECASE)
    workbook_name = normalize_name(workbook_name)
    sheet_component = normalize_name(sheet_name)
    if sheet_component in {"gcsi", "impact_of_crisis", "conditions_of_affected_people", "complexity", "core_indicators", "crisis_indicator_data", "country_indicator_data"}:
        return f"{workbook_name}_{sheet_component}"
    if "inform_severity" in workbook_name or "acaps_inform_severity" in workbook_name:
        return f"inform_severity_{sheet_component}"
    return f"{workbook_name}_{sheet_component}"

# --- Load sheet_manifest_df from the persisted table ---
sheet_manifest_df = spark.table(SHEET_MANIFEST_TABLE)

# --- Analyze data loss from deduplication ---
dataset_to_files = defaultdict(set)
for row in sheet_manifest_df.collect():
    if not is_data_sheet(row["sheet_name"]):
        continue
    dataset_name = classify_excel_dataset(row["file_name"], row["sheet_name"])
    dataset_to_files[dataset_name].add(row["file_name"])

# Show datasets with data loss (more than 1 file collapsed)
print(f"{'DATASET NAME':<60} {'FILES':>6} {'LOST':>6}")
print("-" * 75)
total_lost = 0
for ds, files in sorted(dataset_to_files.items(), key=lambda x: -len(x[1])):
    if len(files) > 1:
        lost = len(files) - 1
        total_lost += lost
        print(f"{ds:<60} {len(files):>6} {lost:>6}")

print(f"\nTotal datasets affected: {sum(1 for f in dataset_to_files.values() if len(f) > 1)}")
print(f"Total workbooks lost: {total_lost}")

In [0]:
%sql
SELECT _source_file_name, COUNT(*) AS row_count
FROM workspace.default.njyoti_bronze_unocha_contributions
GROUP BY _source_file_name

### Checks or severity data files before re-ingesting

In [0]:
# Checking for severity filenames that do not have "gcsi" in the name
import pandas as pd
import re
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed

# --- Idempotent setup ---
SOURCE_ROOT = "/Volumes/cmu_hackathon/common/unocha"
TARGET_SCHEMA = "workspace.default"
TARGET_PREFIX = "njyoti_bronze_unocha"
SHEET_MANIFEST_TABLE = f"{TARGET_SCHEMA}.{TARGET_PREFIX}_sheet_manifest"

def normalize_name(value: str) -> str:
    value = str(value).strip().lower()
    value = re.sub(r"[^a-z0-9]+", "_", value)
    value = re.sub(r"_+", "_", value).strip("_")
    return value or "unnamed"

def to_local_volume_path(path: str) -> str:
    return path.replace("dbfs:/Volumes/", "/Volumes/") if path.startswith("dbfs:/Volumes/") else path

# --- Load sheet manifest ---
sheet_manifest_df = spark.table(SHEET_MANIFEST_TABLE)

# --- Define the 7 target sheet names (normalized) ---
target_sheets = {
    "inform_severity_all_crises",
    "inform_severity_country",
    "impact_of_the_crisis",
    "conditions_of_people_affected",
    "complexity_of_the_crisis",
    "regional_crises",
    "trends"
}

# --- Filter post-rebrand files (no 'gcsi' in filename), only target sheets ---
all_rows = sheet_manifest_df.collect()
sheets_to_check = []
for row in all_rows:
    fn = row["file_name"].lower()
    if "gcsi" in fn:
        continue  # skip pre-rebrand
    sheet_norm = normalize_name(row["sheet_name"])
    if sheet_norm in target_sheets:
        sheets_to_check.append({
            "source_path": row["source_path"],
            "file_name": row["file_name"],
            "sheet_name": row["sheet_name"],
            "sheet_norm": sheet_norm
        })

print(f"Total sheet reads to perform: {len(sheets_to_check)}")
print(f"Files per sheet type:")
from collections import Counter
counts = Counter(r["sheet_norm"] for r in sheets_to_check)
for sheet_type, count in sorted(counts.items()):
    print(f"  {sheet_type}: {count} files")

# --- Read headers only (nrows=0) in parallel ---
def read_header(sheet_info):
    """Read only the header row from an Excel sheet."""
    try:
        local_path = to_local_volume_path(sheet_info["source_path"])
        pdf = pd.read_excel(local_path, sheet_name=sheet_info["sheet_name"], nrows=0, dtype=str)
        columns = [str(col) for col in pdf.columns]
        return {
            "file_name": sheet_info["file_name"],
            "sheet_norm": sheet_info["sheet_norm"],
            "columns": columns,
            "col_count": len(columns),
            "error": None
        }
    except Exception as e:
        return {
            "file_name": sheet_info["file_name"],
            "sheet_norm": sheet_info["sheet_norm"],
            "columns": [],
            "col_count": 0,
            "error": str(e)[:150]
        }

print(f"\nReading headers with 10 parallel workers...")
results = []
with ThreadPoolExecutor(max_workers=10) as executor:
    futures = {executor.submit(read_header, s): s for s in sheets_to_check}
    for future in as_completed(futures):
        results.append(future.result())

print(f"Headers read: {len(results)} ({sum(1 for r in results if r['error'])} errors)")

# --- Analyze column consistency per sheet type ---
print("\n" + "=" * 90)
print("COLUMN CONSISTENCY ANALYSIS PER SHEET TYPE")
print("=" * 90)

for sheet_type in sorted(target_sheets):
    sheet_results = [r for r in results if r["sheet_norm"] == sheet_type and r["error"] is None]
    sheet_errors = [r for r in results if r["sheet_norm"] == sheet_type and r["error"] is not None]
    
    if not sheet_results:
        print(f"\n--- {sheet_type} --- NO SUCCESSFUL READS ({len(sheet_errors)} errors)")
        continue
    
    print(f"\n--- {sheet_type} --- ({len(sheet_results)} files read, {len(sheet_errors)} errors)")
    
    # Column count variation
    col_counts = [r["col_count"] for r in sheet_results]
    print(f"  Column count: min={min(col_counts)}, max={max(col_counts)}, unique counts={sorted(set(col_counts))}")
    
    # Find the "reference" column set (most common)
    col_tuples = [tuple(r["columns"]) for r in sheet_results]
    col_tuple_counts = Counter(col_tuples)
    most_common_cols, most_common_count = col_tuple_counts.most_common(1)[0]
    print(f"  Most common schema appears in {most_common_count}/{len(sheet_results)} files")
    print(f"  Distinct schemas: {len(col_tuple_counts)}")
    
    # Show column differences vs. reference
    reference_set = set(most_common_cols)
    all_columns_seen = set()
    for r in sheet_results:
        all_columns_seen.update(r["columns"])
    
    extra_cols = all_columns_seen - reference_set
    if extra_cols:
        print(f"  Columns in SOME files but not reference: {sorted(extra_cols)[:15]}")
    
    # Check for files that differ from reference
    differing_files = [(r["file_name"], set(r["columns"]) - reference_set, reference_set - set(r["columns"])) 
                       for r in sheet_results if tuple(r["columns"]) != most_common_cols]
    if differing_files:
        print(f"  Files differing from reference ({len(differing_files)}):")
        for fname, added, missing in differing_files[:5]:  # show first 5
            print(f"    {fname}")
            if added:
                print(f"      + added: {sorted(added)[:8]}")
            if missing:
                print(f"      - missing: {sorted(missing)[:8]}")
        if len(differing_files) > 5:
            print(f"    ... and {len(differing_files) - 5} more")
    else:
        print(f"  ✓ All files have IDENTICAL column names")
    
    if sheet_errors:
        print(f"  Errors ({len(sheet_errors)}):")
        for e in sheet_errors[:3]:
            print(f"    {e['file_name']}: {e['error'][:80]}")

In [0]:
# Duplicate detection across 72 post-rebrand INFORM Severity files
# Goal: identify files covering the same data period (month+year) and determine which to keep
import re
from collections import defaultdict

# --- Idempotent setup ---
SOURCE_ROOT = "/Volumes/cmu_hackathon/common/unocha"
TARGET_SCHEMA = "workspace.default"
TARGET_PREFIX = "njyoti_bronze_unocha"
MANIFEST_TABLE = f"{TARGET_SCHEMA}.{TARGET_PREFIX}_manifest"
SHEET_MANIFEST_TABLE = f"{TARGET_SCHEMA}.{TARGET_PREFIX}_sheet_manifest"

def normalize_name(value: str) -> str:
    value = str(value).strip().lower()
    value = re.sub(r"[^a-z0-9]+", "_", value)
    value = re.sub(r"_+", "_", value).strip("_")
    return value or "unnamed"

# --- Load file manifest (has file sizes) and sheet manifest ---
files_manifest_df = spark.table(MANIFEST_TABLE)
sheet_manifest_df = spark.table(SHEET_MANIFEST_TABLE)

# --- Get post-rebrand INFORM Severity files (no 'gcsi' in filename) ---
all_files = files_manifest_df.collect()
inform_files = []
for row in all_files:
    fn = row["file_name"].lower()
    ext = row["extension"].lower() if row["extension"] else ""
    if ext not in {"xls", "xlsx"}:
        continue
    if "gcsi" in fn:
        continue  # pre-rebrand
    if "inform" not in fn and "severity" not in fn:
        continue  # not an inform severity file
    inform_files.append({
        "file_name": row["file_name"],
        "source_path": row["source_path"],
        "size_bytes": row["size_bytes"],
        "extension": ext
    })

print(f"Post-rebrand INFORM Severity files found: {len(inform_files)}")
print()

# --- Extract data period (month + year) from filename ---
MONTH_NAMES = {
    "january": 1, "february": 2, "march": 3, "april": 4,
    "may": 5, "june": 6, "july": 7, "august": 8,
    "september": 9, "october": 10, "november": 11, "december": 12
}

def extract_data_period(file_name: str):
    """Extract the data period (month, year) from the filename.
    Convention: YYYYMMDD-inform-severity-<month>-<year>.xlsx
    The month+year in the name is the DATA period, not the publication date.
    """
    fn_lower = file_name.lower()
    # Try pattern: <month>-<year> or <month>_<year>
    for month_name, month_num in MONTH_NAMES.items():
        # Match month followed by separator and 4-digit year
        pattern = rf"{month_name}[-_](\d{{4}})"
        match = re.search(pattern, fn_lower)
        if match:
            year = int(match.group(1))
            return (year, month_num, month_name.capitalize())
    return None

def extract_publication_date(file_name: str):
    """Extract the publication date prefix (YYYYMMDD or YYYYMM) from filename."""
    match = re.match(r"^(\d{8})[-_]", file_name)
    if match:
        return match.group(1)
    match = re.match(r"^(\d{6})[-_]", file_name)
    if match:
        return match.group(1)
    return None

# --- Group files by data period ---
period_groups = defaultdict(list)
no_period = []

for f in inform_files:
    period = extract_data_period(f["file_name"])
    if period:
        f["data_period"] = period
        f["pub_date"] = extract_publication_date(f["file_name"])
        period_groups[period].append(f)
    else:
        no_period.append(f)

print(f"Files with detected data period: {sum(len(v) for v in period_groups.values())}")
print(f"Files with NO detected period: {len(no_period)}")
if no_period:
    print("  Unmatched files:")
    for f in no_period:
        print(f"    {f['file_name']} ({f['size_bytes']:,} bytes)")

# --- Identify candidate duplicates (periods with 2+ files) ---
print(f"\n{'='*80}")
print("DUPLICATE CANDIDATE ANALYSIS")
print(f"{'='*80}")

duplicates_found = []
for period, files in sorted(period_groups.items()):
    if len(files) > 1:
        year, month_num, month_name = period
        print(f"\n--- {month_name} {year} --- ({len(files)} files)")
        for f in sorted(files, key=lambda x: x["file_name"]):
            print(f"  {f['file_name']}")
            print(f"    Publication date: {f['pub_date']}, Size: {f['size_bytes']:,} bytes")
        
        # Check if sizes match (strong signal for identical content)
        sizes = [f["size_bytes"] for f in files]
        if len(set(sizes)) == 1:
            print(f"  >> IDENTICAL file sizes — very likely duplicate content")
        else:
            size_diff = max(sizes) - min(sizes)
            print(f"  >> DIFFERENT sizes (diff: {size_diff:,} bytes) — likely revision/update")
        
        duplicates_found.append({
            "period": period,
            "files": files,
            "same_size": len(set(sizes)) == 1
        })

if not duplicates_found:
    print("\n✓ No duplicate periods found — all 72 files cover unique months.")
else:
    print(f"\n\nSUMMARY: {len(duplicates_found)} periods with multiple files")
    print(f"  Same size (likely identical): {sum(1 for d in duplicates_found if d['same_size'])}")
    print(f"  Different size (likely revision): {sum(1 for d in duplicates_found if not d['same_size'])}")

# --- Also list all unique periods to verify coverage ---
print(f"\n\nTotal unique data periods covered: {len(period_groups)}")
print(f"Date range: {min(period_groups.keys())} to {max(period_groups.keys())}")

In [0]:
# Content verification for the 2 "different size" duplicate pairs
# Dec 2025: 202512-inform-severity-december-2025.xlsx vs 202512_inform_severity_mid_december_2025.xlsx
# Apr 2026: 202604-inform-severity-april-2026-1.xlsx vs 202604-inform-severity-april-2026.xlsx
import pandas as pd
import hashlib
from concurrent.futures import ThreadPoolExecutor, as_completed

def to_local_volume_path(path: str) -> str:
    return path.replace("dbfs:/Volumes/", "/Volumes/") if path.startswith("dbfs:/Volumes/") else path

SOURCE_ROOT = "/Volumes/cmu_hackathon/common/unocha"

# The pairs to compare (using "INFORM Severity - country" as the comparison sheet — it has real headers)
comparison_sheet = "INFORM Severity - country"

pairs = [
    {
        "period": "December 2025",
        "files": [
            "202512-inform-severity-december-2025.xlsx",
            "202512_inform_severity_mid_december_2025.xlsx"
        ]
    },
    {
        "period": "April 2026",
        "files": [
            "202604-inform-severity-april-2026-1.xlsx",
            "202604-inform-severity-april-2026.xlsx"
        ]
    }
]

def read_sheet_for_comparison(file_name, sheet_name):
    """Read a sheet and return row count, column count, and content hash."""
    try:
        local_path = f"{SOURCE_ROOT}/{file_name}"
        local_path = to_local_volume_path(f"dbfs:{local_path}")
        pdf = pd.read_excel(local_path, sheet_name=sheet_name, dtype=str)
        pdf = pdf.dropna(axis=0, how="all").dropna(axis=1, how="all")
        row_count = len(pdf)
        col_count = len(pdf.columns)
        # Hash the content for comparison
        content_str = pdf.to_csv(index=False)
        content_hash = hashlib.md5(content_str.encode()).hexdigest()
        # Get first and last few values for inspection
        first_rows = pdf.head(3).to_string()
        last_rows = pdf.tail(3).to_string()
        return {
            "file_name": file_name,
            "row_count": row_count,
            "col_count": col_count,
            "content_hash": content_hash,
            "columns": list(pdf.columns[:10]),
            "first_rows": first_rows,
            "last_rows": last_rows,
            "error": None
        }
    except Exception as e:
        return {
            "file_name": file_name,
            "row_count": 0,
            "col_count": 0,
            "content_hash": None,
            "columns": [],
            "first_rows": "",
            "last_rows": "",
            "error": str(e)[:200]
        }

print("=" * 80)
print("CONTENT COMPARISON FOR DIFFERENT-SIZE DUPLICATE PAIRS")
print("=" * 80)

for pair in pairs:
    print(f"\n--- {pair['period']} ---")
    print(f"  Comparison sheet: '{comparison_sheet}'")
    
    results = []
    for fname in pair["files"]:
        result = read_sheet_for_comparison(fname, comparison_sheet)
        results.append(result)
    
    for r in results:
        if r["error"]:
            print(f"  {r['file_name']}: ERROR - {r['error']}")
        else:
            print(f"  {r['file_name']}:")
            print(f"    Rows: {r['row_count']}, Cols: {r['col_count']}")
            print(f"    Hash: {r['content_hash']}")
            print(f"    Columns (first 10): {r['columns']}")
    
    # Compare
    if all(r["error"] is None for r in results):
        if results[0]["content_hash"] == results[1]["content_hash"]:
            print(f"  >> IDENTICAL content (same hash) despite different file sizes")
            print(f"     (File size difference likely from Excel metadata/formatting only)")
        else:
            print(f"  >> DIFFERENT content:")
            print(f"     Row diff: {results[0]['row_count']} vs {results[1]['row_count']}")
            print(f"     Col diff: {results[0]['col_count']} vs {results[1]['col_count']}")
            # Show which has more data
            if results[0]["row_count"] > results[1]["row_count"]:
                print(f"     >>> '{results[0]['file_name']}' has MORE rows — likely the fuller version")
            elif results[1]["row_count"] > results[0]["row_count"]:
                print(f"     >>> '{results[1]['file_name']}' has MORE rows — likely the fuller version")
            else:
                print(f"     >>> Same row count but different content — column changes or data corrections")

# --- Final exclusion list based on all findings ---
print("\n\n" + "=" * 80)
print("FINAL EXCLUSION LIST FOR INGESTION")
print("=" * 80)
print("\nBased on duplicate detection + content verification:")

# Identical-size pairs: exclude the '(1)' copy (download artifact)
# Different-size pairs: keep the standard naming convention (no 'mid_', no '-1' suffix)
FILES_TO_EXCLUDE = {
    # Identical-size duplicates (download artifacts with ' (1)' in name)
    "202509_inform_severity_-_september_2025 (1).xlsx",
    "202602-inform-severity-february-2026-1 (1).xlsx",
    "202603-inform-severity-march-2026-1 (1).xlsx",
    # Different-size pairs: exclude the non-standard/interim version
    "202512_inform_severity_mid_december_2025.xlsx",   # interim 'mid-december' release
    "202604-inform-severity-april-2026-1.xlsx",        # '-1' suffix = superseded by final
}

print(f"\nFiles to EXCLUDE from ingestion ({len(FILES_TO_EXCLUDE)}):")
for f in sorted(FILES_TO_EXCLUDE):
    print(f"  ✗ {f}")

print(f"\nFiles to KEEP for ingestion: {73 - len(FILES_TO_EXCLUDE)} → 68 unique months")

In [0]:
# Final exclusion list and deduplicated file list for ingestion
# Decision: keep '-1' version for Apr 2026 (has more rows: 73 vs 71)
import re

SOURCE_ROOT = "/Volumes/cmu_hackathon/common/unocha"
TARGET_SCHEMA = "workspace.default"
TARGET_PREFIX = "njyoti_bronze_unocha"
MANIFEST_TABLE = f"{TARGET_SCHEMA}.{TARGET_PREFIX}_manifest"

FILES_TO_EXCLUDE = {
    # Identical-size duplicates (download artifacts with ' (1)' in name)
    "202509_inform_severity_-_september_2025 (1).xlsx",
    "202602-inform-severity-february-2026-1 (1).xlsx",
    "202603-inform-severity-march-2026-1 (1).xlsx",
    # Different-size pairs:
    "202512_inform_severity_mid_december_2025.xlsx",   # interim 'mid-december' release; keep final
    "202604-inform-severity-april-2026.xlsx",          # fewer rows (71 vs 73); keep '-1' version
}

# --- Generate the final list of unique files to ingest ---
files_manifest_df = spark.table(MANIFEST_TABLE)
all_files = files_manifest_df.collect()

inform_files_to_ingest = []
for row in all_files:
    fn = row["file_name"]
    fn_lower = fn.lower()
    ext = row["extension"].lower() if row["extension"] else ""
    if ext not in {"xls", "xlsx"}:
        continue
    if "gcsi" in fn_lower:
        continue  # pre-rebrand
    if "inform" not in fn_lower and "severity" not in fn_lower:
        continue  # not an inform severity file
    if fn in FILES_TO_EXCLUDE:
        continue  # excluded duplicate
    inform_files_to_ingest.append(fn)

inform_files_to_ingest.sort()

print("=" * 80)
print("FINAL DEDUPLICATED FILE LIST FOR INGESTION")
print("=" * 80)
print(f"\nFiles excluded: {len(FILES_TO_EXCLUDE)}")
for f in sorted(FILES_TO_EXCLUDE):
    print(f"  \u2717 {f}")

print(f"\nTotal files to ingest: {len(inform_files_to_ingest)}")
print()
for i, f in enumerate(inform_files_to_ingest, 1):
    print(f"  {i:>3}. {f}")

## Re-Ingestion of Severity Data

In [0]:
# =============================================================================
# RE-INGESTION: Post-rebrand INFORM Severity files (Option C)
# - Reads 7 target sheets from 68 deduplicated post-rebrand workbooks
# - Unions all workbooks per sheet type into a single bronze table
# - Uses parallel workers for speed
# - Produces 7 new bronze tables with clean naming (no _stage_ prefix)
# =============================================================================
import pandas as pd
import re
import time
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor, as_completed
from pyspark.sql import functions as F
from pyspark.sql import types as T

# --- Constants ---
SOURCE_ROOT = "/Volumes/cmu_hackathon/common/unocha"
TARGET_SCHEMA = "workspace.default"
TARGET_PREFIX = "njyoti_bronze_unocha"
MANIFEST_TABLE = f"{TARGET_SCHEMA}.{TARGET_PREFIX}_manifest"
SHEET_MANIFEST_TABLE = f"{TARGET_SCHEMA}.{TARGET_PREFIX}_sheet_manifest"

def normalize_name(value: str) -> str:
    value = str(value).strip().lower()
    value = re.sub(r"[^a-z0-9]+", "_", value)
    value = re.sub(r"_+", "_", value).strip("_")
    return value or "unnamed"

def to_local_volume_path(path: str) -> str:
    return path.replace("dbfs:/Volumes/", "/Volumes/") if path.startswith("dbfs:/Volumes/") else path

def deduplicate_columns(column_names):
    normalized_cols = []
    seen = set()
    for column_name in column_names:
        normalized = normalize_name(column_name)
        candidate = normalized
        suffix = 1
        while candidate in seen:
            suffix += 1
            candidate = f"{normalized}_{suffix}"
        normalized_cols.append(candidate)
        seen.add(candidate)
    return normalized_cols

# --- Files to exclude (from cell 22 analysis) ---
FILES_TO_EXCLUDE = {
    # Identical-size duplicates (download artifacts)
    "202509_inform_severity_-_september_2025 (1).xlsx",
    "202602-inform-severity-february-2026-1 (1).xlsx",
    "202603-inform-severity-march-2026-1 (1).xlsx",
    # Different-size pairs
    "202512_inform_severity_mid_december_2025.xlsx",   # interim release; keep final
    "202604-inform-severity-april-2026.xlsx",          # fewer rows; keep '-1' version
    # Pre-rebrand file with incompatible sheet names (GCSI-era structure)
    "20200403-acaps-inform-severity-index_dataset-march-2020.xlsx",
}

# --- Target sheets to ingest (normalized names) ---
TARGET_SHEETS = {
    "inform_severity_all_crises",
    "inform_severity_country",
    "impact_of_the_crisis",
    "conditions_of_people_affected",
    "complexity_of_the_crisis",
    "regional_crises",
    "trends",
}

# --- Map from normalized sheet name to bronze table name ---
SHEET_TO_TABLE = {
    "inform_severity_all_crises": f"{TARGET_SCHEMA}.{TARGET_PREFIX}_inform_severity_all_crises",
    "inform_severity_country": f"{TARGET_SCHEMA}.{TARGET_PREFIX}_inform_severity_country",
    "impact_of_the_crisis": f"{TARGET_SCHEMA}.{TARGET_PREFIX}_inform_severity_impact_of_the_crisis",
    "conditions_of_people_affected": f"{TARGET_SCHEMA}.{TARGET_PREFIX}_inform_severity_conditions_of_people_affected",
    "complexity_of_the_crisis": f"{TARGET_SCHEMA}.{TARGET_PREFIX}_inform_severity_complexity_of_the_crisis",
    "regional_crises": f"{TARGET_SCHEMA}.{TARGET_PREFIX}_inform_severity_regional_crises",
    "trends": f"{TARGET_SCHEMA}.{TARGET_PREFIX}_inform_severity_trends",
}

# --- Build the list of (file_name, source_path, sheet_name, sheet_norm) to read ---
sheet_manifest_df = spark.table(SHEET_MANIFEST_TABLE)
all_sheet_rows = sheet_manifest_df.collect()

sheets_to_ingest = []  # list of dicts
for row in all_sheet_rows:
    fn = row["file_name"]
    if fn in FILES_TO_EXCLUDE:
        continue
    if "gcsi" in fn.lower():
        continue  # pre-rebrand family
    ext = fn.rsplit(".", 1)[-1].lower() if "." in fn else ""
    if ext not in {"xls", "xlsx"}:
        continue
    if "inform" not in fn.lower() and "severity" not in fn.lower():
        continue
    sheet_norm = normalize_name(row["sheet_name"])
    if sheet_norm not in TARGET_SHEETS:
        continue
    sheets_to_ingest.append({
        "file_name": fn,
        "source_path": row["source_path"],
        "sheet_name": row["sheet_name"],
        "sheet_norm": sheet_norm,
    })

print(f"Total sheet reads to perform: {len(sheets_to_ingest)}")
from collections import Counter
counts = Counter(s["sheet_norm"] for s in sheets_to_ingest)
for sheet_type, count in sorted(counts.items()):
    print(f"  {sheet_type}: {count} files")

# --- Parallel reader: read one sheet, return pandas DataFrame + metadata ---
def read_one_sheet(sheet_info):
    """Read a single Excel sheet and return (pandas_df, metadata) or (None, error)."""
    try:
        local_path = to_local_volume_path(sheet_info["source_path"])
        pdf = pd.read_excel(local_path, sheet_name=sheet_info["sheet_name"], dtype=str)
        if pdf.empty:
            return None
        pdf = pdf.dropna(axis=0, how="all").dropna(axis=1, how="all")
        if pdf.empty:
            return None
        # Normalize column names
        pdf.columns = deduplicate_columns([str(col) for col in pdf.columns])
        # Replace NaN with None for Spark compatibility
        pdf = pdf.where(pd.notnull(pdf), None)
        return {
            "pdf": pdf,
            "file_name": sheet_info["file_name"],
            "source_path": sheet_info["source_path"],
            "sheet_name": sheet_info["sheet_name"],
            "sheet_norm": sheet_info["sheet_norm"],
            "error": None,
        }
    except Exception as e:
        return {
            "pdf": None,
            "file_name": sheet_info["file_name"],
            "source_path": sheet_info["source_path"],
            "sheet_name": sheet_info["sheet_name"],
            "sheet_norm": sheet_info["sheet_norm"],
            "error": str(e)[:200],
        }

# --- Read all sheets in parallel ---
print(f"\nReading {len(sheets_to_ingest)} sheets with 10 parallel workers...")
start_time = time.time()

read_results = []  # list of dicts from read_one_sheet
with ThreadPoolExecutor(max_workers=10) as executor:
    futures = {executor.submit(read_one_sheet, s): s for s in sheets_to_ingest}
    for future in as_completed(futures):
        result = future.result()
        if result is not None:
            read_results.append(result)

elapsed = time.time() - start_time
successes = [r for r in read_results if r["error"] is None and r["pdf"] is not None]
failures = [r for r in read_results if r["error"] is not None]
skipped = len(sheets_to_ingest) - len(read_results)

print(f"Read complete in {elapsed:.1f}s")
print(f"  Successful: {len(successes)}")
print(f"  Failed: {len(failures)}")
print(f"  Skipped (empty): {skipped}")

if failures:
    print("\n  Failed sheets:")
    for f in failures[:10]:
        print(f"    {f['file_name']}::{f['sheet_name']}: {f['error'][:80]}")
    if len(failures) > 10:
        print(f"    ... and {len(failures) - 10} more")

# --- Group successful reads by sheet type and union into Spark DataFrames ---
print(f"\n{'='*80}")
print("WRITING BRONZE TABLES")
print(f"{'='*80}")

sheets_by_type = defaultdict(list)
for r in successes:
    sheets_by_type[r["sheet_norm"]].append(r)

ingestion_summary = []

for sheet_norm in sorted(TARGET_SHEETS):
    table_name = SHEET_TO_TABLE[sheet_norm]
    sheet_data = sheets_by_type.get(sheet_norm, [])
    
    if not sheet_data:
        print(f"\n  {sheet_norm}: NO DATA — skipping")
        continue
    
    print(f"\n  {sheet_norm}: unioning {len(sheet_data)} DataFrames...")
    
    # Convert each pandas DF to Spark DF with metadata, then union
    bronze_df = None
    for r in sheet_data:
        pdf = r["pdf"]
        # All columns as StringType for bronze
        schema = T.StructType([T.StructField(col, T.StringType(), True) for col in pdf.columns])
        rows = [tuple(None if v is None else str(v) for v in row) for row in pdf.itertuples(index=False, name=None)]
        sdf = spark.createDataFrame(rows, schema=schema)
        # Attach metadata columns
        sdf = (
            sdf
            .withColumn("_source_file", F.lit(r["source_path"]))
            .withColumn("_source_file_name", F.lit(r["file_name"]))
            .withColumn("_sheet_name", F.lit(r["sheet_name"]))
            .withColumn("_dataset_name", F.lit(sheet_norm))
            .withColumn("_ingest_ts", F.current_timestamp())
        )
        if bronze_df is None:
            bronze_df = sdf
        else:
            bronze_df = bronze_df.unionByName(sdf, allowMissingColumns=True)
    
    # Write the unioned table
    row_count = bronze_df.count()
    bronze_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(table_name)
    
    print(f"    \u2713 {table_name} — {row_count:,} rows from {len(sheet_data)} workbooks")
    ingestion_summary.append({
        "sheet_type": sheet_norm,
        "table_name": table_name,
        "source_files": len(sheet_data),
        "row_count": row_count,
    })

# --- Summary ---
print(f"\n\n{'='*80}")
print("INGESTION SUMMARY")
print(f"{'='*80}")
print(f"\n{'Sheet Type':<45} {'Table':<70} {'Files':>6} {'Rows':>10}")
print("-" * 135)
for s in ingestion_summary:
    print(f"{s['sheet_type']:<45} {s['table_name']:<70} {s['source_files']:>6} {s['row_count']:>10,}")
print(f"\nTotal elapsed time: {time.time() - start_time:.1f}s")

In [0]:
# =============================================================================
# CLEANUP: Drop old problematic _stage_inform_severity_* tables
# These contain data from only 1 workbook (the dedup bug). They are now
# superseded by the 7 new union tables written above.
# 
# SAFE: Does NOT touch CSV tables, pre-rebrand GCSI stage tables, or manifests.
# =============================================================================

TARGET_SCHEMA = "workspace.default"
TARGET_PREFIX = "njyoti_bronze_unocha"
MANIFEST_TABLE = f"{TARGET_SCHEMA}.{TARGET_PREFIX}_manifest"
SHEET_MANIFEST_TABLE = f"{TARGET_SCHEMA}.{TARGET_PREFIX}_sheet_manifest"

# Tables to preserve (never drop these)
PROTECTED_TABLES = {
    MANIFEST_TABLE.split(".")[-1],
    SHEET_MANIFEST_TABLE.split(".")[-1],
}

# Find all old stage tables that contain 'inform_severity' but NOT 'gcsi'
existing_tables = spark.sql(f"SHOW TABLES IN {TARGET_SCHEMA}").collect()

tables_to_drop = []
for tbl in existing_tables:
    tbl_name = tbl["tableName"]
    # Only target stage tables from the old INFORM Severity ingestion
    if not tbl_name.startswith(f"{TARGET_PREFIX}_stage_"):
        continue
    if "inform_severity" not in tbl_name:
        continue
    # Don't drop pre-rebrand/GCSI stage tables
    if "gcsi" in tbl_name:
        continue
    # Don't drop protected tables
    if tbl_name in PROTECTED_TABLES:
        continue
    tables_to_drop.append(tbl_name)

print(f"Old stage tables to drop: {len(tables_to_drop)}")
print()

for tbl_name in sorted(tables_to_drop):
    full_name = f"{TARGET_SCHEMA}.{tbl_name}"
    spark.sql(f"DROP TABLE IF EXISTS {full_name}")
    print(f"  \u2717 Dropped: {full_name}")

print(f"\n\u2713 Cleanup complete. {len(tables_to_drop)} old tables removed.")
print(f"  New bronze tables (from cell above) are now the authoritative source.")

In [0]:
# =============================================================================
# SUMMARIZE BRONZE OUTPUTS (post re-ingestion)
# Scans all tables in workspace.default matching the bronze prefix.
# Uses DESCRIBE DETAIL for fast row counts (reads Delta log, no full scan).
# =============================================================================
from pyspark.sql import functions as F, types as T

TARGET_SCHEMA = "workspace.default"
TARGET_PREFIX = "njyoti_bronze_unocha"
MANIFEST_TABLE = f"{TARGET_SCHEMA}.{TARGET_PREFIX}_manifest"
SHEET_MANIFEST_TABLE = f"{TARGET_SCHEMA}.{TARGET_PREFIX}_sheet_manifest"

# Tables to skip in the summary (infrastructure, not data)
skip_suffixes = {"_manifest", "_sheet_manifest", "_excel_selected_sheets"}

# Known INFORM Severity union tables (from cell 24 re-ingestion)
INFORM_SEVERITY_TABLES = {
    f"{TARGET_PREFIX}_inform_severity_all_crises",
    f"{TARGET_PREFIX}_inform_severity_country",
    f"{TARGET_PREFIX}_inform_severity_impact_of_the_crisis",
    f"{TARGET_PREFIX}_inform_severity_conditions_of_people_affected",
    f"{TARGET_PREFIX}_inform_severity_complexity_of_the_crisis",
    f"{TARGET_PREFIX}_inform_severity_regional_crises",
    f"{TARGET_PREFIX}_inform_severity_trends",
}

# --- Scan all tables ---
all_tables = {row["tableName"] for row in spark.sql(f"SHOW TABLES IN {TARGET_SCHEMA}").collect()}

summary_rows = []
for tbl in sorted(all_tables):
    if not tbl.startswith(TARGET_PREFIX):
        continue
    if any(tbl.endswith(s) for s in skip_suffixes):
        continue
    
    full_name = f"{TARGET_SCHEMA}.{tbl}"
    dataset_name = tbl.replace(f"{TARGET_PREFIX}_", "", 1)
    
    # Categorize
    if tbl in INFORM_SEVERITY_TABLES:
        source_type = "excel_inform_severity"
    elif "_stage_" in tbl and "gcsi" in tbl:
        source_type = "excel_stage_gcsi"
    elif "_stage_" in tbl:
        source_type = "excel_stage_other"
    else:
        source_type = "csv"
    
    # Get row count from Delta log (fast — no full table scan)
    try:
        detail = spark.sql(f"DESCRIBE DETAIL {full_name}").collect()[0]
        row_count = detail["numRecords"] if detail["numRecords"] is not None else -1
    except Exception:
        row_count = -1
    
    summary_rows.append({
        "dataset_name": dataset_name,
        "table_name": full_name,
        "source_type": source_type,
        "record_count": row_count,
    })

# --- Create summary DataFrame ---
summary_schema = T.StructType([
    T.StructField("dataset_name", T.StringType(), True),
    T.StructField("table_name", T.StringType(), True),
    T.StructField("source_type", T.StringType(), True),
    T.StructField("record_count", T.LongType(), True),
])

bronze_summary_df = spark.createDataFrame(summary_rows, schema=summary_schema)

# --- Print summary by source type ---
print("=" * 80)
print("BRONZE LAYER SUMMARY (post re-ingestion)")
print("=" * 80)

total_tables = len(summary_rows)
total_rows = sum(r["record_count"] for r in summary_rows if r["record_count"] > 0)

print(f"\nTotal bronze tables: {total_tables}")
print(f"Total rows across all tables: {total_rows:,}")

# Group by source type
from collections import defaultdict, Counter
type_counts = Counter(r["source_type"] for r in summary_rows)
type_rows = defaultdict(int)
for r in summary_rows:
    if r["record_count"] > 0:
        type_rows[r["source_type"]] += r["record_count"]

print(f"\n{'Source Type':<25} {'Tables':>8} {'Total Rows':>15}")
print("-" * 50)
for stype in ["csv", "excel_inform_severity", "excel_stage_gcsi", "excel_stage_other"]:
    if type_counts.get(stype, 0) > 0:
        print(f"{stype:<25} {type_counts[stype]:>8} {type_rows[stype]:>15,}")

# --- Display full table ---
print(f"\n\n{'Dataset':<65} {'Type':<25} {'Rows':>12}")
print("-" * 105)
for r in sorted(summary_rows, key=lambda x: (x["source_type"], x["dataset_name"])):
    print(f"{r['dataset_name']:<65} {r['source_type']:<25} {r['record_count']:>12,}")

# Also display as a table for easier inspection
display(bronze_summary_df.orderBy("source_type", "dataset_name"))